# Predicción de defectos de software mediante Machine Learning
## Desarrollo y evaluación experimental del modelo predictivo

### Descripción del proyecto

Este notebook implementa el componente experimental de una investigación orientada a la **predicción de defectos de software mediante algoritmos de Machine Learning**, con el propósito de apoyar las actividades de aseguramiento de calidad (QA) a través de la identificación y priorización de módulos con mayor propensión a presentar defectos.

El experimento adopta el enfoque **Cross-Project Defect Prediction (CPDP)**, que permite evaluar la capacidad de los modelos para realizar predicciones en proyectos de software diferentes de aquellos utilizados durante su entrenamiento.

El desarrollo del componente predictivo sigue las fases de **CRISP-DM**, mientras que su evaluación emplea una estrategia de validación cruzada anidada **Leave-One-Project-Out (Nested LOPO)**, reservando proyectos completos para comprobar la capacidad de generalización de los algoritmos.

### Objetivo del experimento

**Comparar el desempeño de los algoritmos SVM, Random Forest y XGBoost**, considerando nueve escenarios de preparación de datos que incorporan técnicas de balanceo de clases y selección de características, con el fin de identificar una configuración predictiva adecuada para su integración en una aplicación web de apoyo al aseguramiento de calidad del software.

### Datos y metodología

Se utilizan cinco conjuntos de datos históricos de NASA/PROMISE: **CM1, JM1, KC1, KC2 y PC1**, representados mediante 21 métricas estáticas de software.

El procedimiento experimental comprende la adquisición de datos, auditoría, normalización de características, entrenamiento de modelos, validación entre proyectos, comparación de métricas y selección de una configuración final.

El desempeño se analiza principalmente mediante **F1-score**, utilizando Average Precision y AUC-ROC como criterios complementarios. También se evalúa la capacidad de los modelos para priorizar módulos mediante su comparación con estrategias de referencia.

### Organización del notebook

El desarrollo se estructura en seis módulos:

1. **M01 — Extracción de datos:** adquisición de datasets públicos y registro de fuentes.
2. **M02 — Preprocesamiento y auditoría:** verificación de características, etiquetas, calidad y compatibilidad de los datos.
3. **M03 — Modelado:** configuración de los algoritmos y escenarios experimentales.
4. **M04 — Validación entre proyectos:** entrenamiento, selección y evaluación mediante Nested LOPO.
5. **M05 — Evaluación y modelo final:** comparación de resultados, selección de la configuración y entrenamiento del pipeline definitivo.
6. **M06 — Exportación:** generación de modelos serializados, métricas, gráficos y evidencias para su integración y documentación.

### Resultado esperado

Obtener una **configuración predictiva seleccionada mediante evidencia experimental**, entrenar un pipeline final con los datos históricos disponibles y exportarlo para su integración en una aplicación web desarrollada con **Streamlit**.

La aplicación permitirá cargar archivos CSV con métricas compatibles, estimar la propensión a defectos y generar un ranking de módulos que apoye la planificación de actividades de revisión y pruebas de software.

### Alcance y consideraciones

El experimento es independiente de ejecuciones anteriores y descarga los datasets públicos en un directorio propio, sin reutilizar modelos previamente entrenados.

Los datos provienen de un repositorio espejo de NASA/PROMISE y requieren verificación documental de su procedencia y etiquetas. Los resultados de validación entre proyectos permiten estudiar la generalización dentro del conjunto evaluado, pero no garantizan un rendimiento equivalente en cualquier proyecto externo.

El modelo final se entrena con los proyectos disponibles después de la selección experimental; las métricas obtenidas mediante validación cruzada corresponden al procedimiento de evaluación y no constituyen una prueba independiente adicional del archivo final serializado.

## Preparación — Instalar dependencias
Ejecuta esta celda y, si Colab pide reiniciar el entorno, vuelve a ejecutar desde esta sección. No uses archivos ni carpetas anteriores.

In [ ]:
%pip -q install "numpy>=1.26,<3" "pandas>=2.1,<3" "scikit-learn>=1.5,<2" "imbalanced-learn>=0.13,<0.15" "xgboost>=2,<4" "joblib>=1.4,<2" "matplotlib>=3.8,<4"

## M01 — EXTRACTOR
Crea una **carpeta temporal completamente nueva**, descarga `CM1.csv`, `JM1.csv`, `KC1.csv`, `KC2.csv`, `PC1.csv` desde el repositorio público y guarda su procedencia y hashes SHA-256. No lee ni reutiliza resultados anteriores.

Fuente del espejo público: `https://github.com/ApoorvaKrisna/NASA-promise-dataset-repository`. Su procedencia y definición de defectos deben revisarse documentalmente para el informe final.

In [ ]:
from pathlib import Path
import tempfile, sys, json, importlib, warnings
import pandas as pd
import numpy as np
from IPython.display import display
BASE = Path(tempfile.mkdtemp(prefix='T2_CPDP_DESDE_CERO_', dir='/content' if Path('/content').exists() else None))
RAW = BASE/'01_datos_originales'
AUDIT = BASE/'02_auditoria'
EXP = BASE/'03_experimentos'
FINAL = BASE/'04_modelo_final'
EVID = BASE/'05_evidencias'
for p in (RAW,AUDIT,EXP,FINAL,EVID): p.mkdir(parents=True,exist_ok=True)
print('NUEVO experimento, sin acceder a proyectos anteriores:',BASE)

In [ ]:
CORE_PY = BASE/'cpdp_core_desde_cero.py'
CORE_PY.write_text('"""PDS tesis — validación ENTRE proyectos NASA/PROMISE, nueva ejecución completamente independiente.\n\nExperimento educativo auditable: no es garantía de defectos de un proyecto nuevo.\nÚnicamente métricas históricas compatibles, no extrae código fuente.\n"""\nfrom __future__ import annotations\nimport hashlib, json, os, platform, sys, time\nfrom datetime import datetime, timezone\nfrom functools import partial\nfrom pathlib import Path\nfrom importlib.metadata import version\nimport numpy as np\nimport pandas as pd\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.feature_selection import SelectKBest, mutual_info_classif, SelectFromModel\nfrom sklearn.svm import SVC\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.metrics import (f1_score, recall_score, precision_score, accuracy_score,\n    average_precision_score, roc_auc_score, confusion_matrix)\nfrom imblearn.pipeline import Pipeline\nfrom imblearn.over_sampling import SMOTE, ADASYN\nfrom xgboost import XGBClassifier\nimport joblib\n\nSEED = 42\nEXPERIMENT_VERSION = \'CPDP_INDEPENDIENTE_V11_NESTED_LOGO_20261011\'\nPROJECTS = (\'CM1\',\'JM1\',\'KC1\',\'KC2\',\'PC1\')\nLABELS = {\'CM1\':\'defects\',\'JM1\':\'defects\',\'KC1\':\'defects\',\'KC2\':\'problems\',\'PC1\':\'defects\'}\nSOURCE_BASE = \'https://raw.githubusercontent.com/ApoorvaKrisna/NASA-promise-dataset-repository/main\'\n# Orden de las 21 métricas; preservado en la exportación e inferencia. Se valida antes de unir.\nFEATURES = (\n    \'loc\',\'v(g)\',\'ev(g)\',\'iv(g)\',\'n\',\'v\',\'l\',\'d\',\'i\',\'e\',\'b\',\'t\',\n    \'lOCode\',\'lOComment\',\'lOBlank\',\'locCodeAndComment\',\n    \'uniq_Op\',\'uniq_Opnd\',\'total_Op\',\'total_Opnd\',\'branchCount\'\n)\n# Alias explícito a documentar. OJO: no implica equivalencia verificada por el autor.\nALIASES = {\'locodeandcomment\':\'loccodeandcomment\'}\nSCENARIOS = (\n    (\'E0_BASELINE\',\'NONE\',\'NONE\'),\n    (\'E1_SMOTE\',\'SMOTE\',\'NONE\'),\n    (\'E2_ADASYN\',\'ADASYN\',\'NONE\'),\n    (\'E3_MI\',\'NONE\',\'MI\'),\n    (\'E4_RF_FS\',\'NONE\',\'RF\'),\n    (\'E5_SMOTE_MI\',\'SMOTE\',\'MI\'),\n    (\'E6_ADASYN_MI\',\'ADASYN\',\'MI\'),\n    (\'E7_SMOTE_RF\',\'SMOTE\',\'RF\'),\n    (\'E8_ADASYN_RF\',\'ADASYN\',\'RF\'),\n)\nALGORITHMS = (\'SVM\',\'Random Forest\',\'XGBoost\')\nCANDIDATES = tuple((s[0],a) for s in SCENARIOS for a in ALGORITHMS)\n\n\ndef sha256(path):\n    h = hashlib.sha256()\n    with open(path,\'rb\') as f:\n        for part in iter(lambda:f.read(1024*1024),b\'\'):\n            h.update(part)\n    return h.hexdigest()\n\n\ndef jdump(path,obj):\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\n    tmp=path.with_suffix(path.suffix+\'.tmp\')\n    tmp.write_text(json.dumps(obj,indent=2,ensure_ascii=False,allow_nan=False,\n        default=lambda x:x.item() if isinstance(x,np.generic) else str(x)),encoding=\'utf-8\')\n    os.replace(tmp,path)\n\n\ndef standard_key(s):\n    return str(s).strip().casefold()\n\n\ndef harmonize_features(raw,project,*, allow_extra=False):\n    """Mapeo verificable por encabezados, nunca descartar columnas silenciosamente."""\n    raw=raw.copy()\n    found={}; alias_logs=[]\n    for col in raw.columns:\n        k=standard_key(col)\n        if k in ALIASES:\n            alias_logs.append({\'proyecto\':project,\'columna_original\':str(col),\n                \'equivalencia_propuesta\':\'locCodeAndComment\',\'pendiente_verificacion_documental\':True})\n            k=ALIASES[k]\n        if k in found:\n            raise ValueError(f\'{project}: colisión tras normalizar columnas {found[k]} y {col}\')\n        found[k]=col\n    desired={standard_key(x):x for x in FEATURES}\n    missing=sorted(set(desired)-set(found))\n    extras=sorted(set(found)-set(desired))\n    if missing or (extras and not allow_extra):\n        raise ValueError(f\'{project}: esquema incompatible: faltantes={missing}, extras={extras}\')\n    out=pd.DataFrame(index=raw.index)\n    for norm,canonical in desired.items():\n        vals=raw[found[norm]].replace([\'?\',\'NA\',\'NaN\',\'\'],np.nan)\n        num=pd.to_numeric(vals,errors=\'coerce\')\n        invalid=vals.notna() & num.isna()\n        if invalid.any():\n            raise ValueError(f\'{project}: caracteres no numéricos en {found[norm]}\')\n        if np.isinf(num.to_numpy(dtype=float,na_value=np.nan)).any():\n            raise ValueError(f\'{project}: valores infinitos en {found[norm]}\')\n        out[canonical]=num.astype(float)\n    if out.empty: raise ValueError(f\'{project}: dataset vacío\')\n    return out,alias_logs\n\n\ndef load_project(path,name):\n    """Solo admite etiqueta histórica conocida por proyecto y 21 métricas esperadas."""\n    if name not in PROJECTS: raise ValueError(name)\n    raw=pd.read_csv(path,low_memory=False)\n    if raw.empty or raw.columns.duplicated().any(): raise ValueError(f\'{name}: CSV vacío o cabeceras repetidas\')\n    cols=[c for c in raw if standard_key(c)==LABELS[name]]\n    if len(cols)!=1: raise ValueError(f\'{name}: etiqueta esperada={LABELS[name]}, detectadas={list(raw.columns)}\')\n    target=cols[0]\n    labels=raw[target].astype(\'string\').str.strip().str.casefold()\n    mapping={\'false\':0,\'true\':1,\'0\':0,\'1\':1,\'0.0\':0,\'1.0\':1,\'no\':0,\'yes\':1}\n    unexpected=sorted(set(labels.dropna())-set(mapping))\n    if unexpected or labels.isna().any():\n        raise ValueError(f\'{name}: valores de etiqueta desconocidos={unexpected}\')\n    y=labels.map(mapping).astype(int).reset_index(drop=True)\n    if y.nunique()!=2: raise ValueError(f\'{name}: se necesitan ambas clases\')\n    X,alias_logs=harmonize_features(raw.drop(columns=[target]),name)\n    X=X.reset_index(drop=True)\n    fingerprint=pd.util.hash_pandas_object(X,index=False)\n    audit={\'proyecto\':name,\'filas\':len(y),\'positivos\':int(y.sum()),\'negativos\':int((y==0).sum()),\n        \'prevalencia\':float(y.mean()),\'variables\':len(FEATURES),\n        \'etiqueta_original\':target,\'etiqueta_binaria\':\'1=defecto_historico_registrado\',\n        \'sha256\':sha256(path),\'duplicados_fila_completa\':int(raw.duplicated().sum()),\n        \'duplicados_vector_caracteristicas\':int(fingerprint.duplicated().sum()),\n        \'faltantes\':int(X.isna().sum().sum()),\'alias_pendientes\':alias_logs,\n        \'columnas_originales\':list(map(str,raw.columns)),\n        \'procedencia_declarada\':f\'{SOURCE_BASE}/{name.lower()}.csv\',\n        \'verificacion_documental_realizada\':False}\n    return X,y,audit\n\n\ndef load_all(raw_dir,names=PROJECTS):\n    data={};audits=[]\n    for name in names:\n        p=Path(raw_dir)/f\'{name}.csv\'\n        if not p.exists(): raise FileNotFoundError(f\'{name}: falta {p}\')\n        x,y,a=load_project(p,name)\n        data[name]=(x,y)\n        audits.append(a)\n    return data,audits\n\n\ndef overlap_audit(data):\n    hashes={};pairs=[]\n    for p,(x,y) in data.items():\n        hashes[p]=pd.util.hash_pandas_object(x,index=False).to_numpy(dtype=np.uint64)\n    for i,p in enumerate(data):\n        for q in list(data)[i+1:]:\n            hs1=set(hashes[p].tolist());hs2=set(hashes[q].tolist());common=hs1&hs2\n            pairs.append({\'proyecto_a\':p,\'proyecto_b\':q,\'vectores_identicos\':len(common),\n               \'filas_de_a_coincidentes\':int(np.isin(hashes[p],list(common)).sum()),\n               \'filas_de_b_coincidentes\':int(np.isin(hashes[q],list(common)).sum())})\n    return pd.DataFrame(pairs)\n\n\ndef pack(data,projects):\n    """Cada ejemplo pertenece a su proyecto; sin derivar información de ningún test."""\n    return (pd.concat([data[p][0] for p in projects],ignore_index=True),\n            pd.concat([data[p][1] for p in projects],ignore_index=True))\n\n\ndef make_pipeline(scenario_id,algorithm):\n    _,balance,select=next(v for v in SCENARIOS if v[0]==scenario_id)\n    # Selectores y balanceadores DENTRO del pipeline: se ajustan solo en training.\n    if select==\'MI\':\n        selector=SelectKBest(partial(mutual_info_classif,random_state=SEED), k=10)\n    elif select==\'RF\':\n        selector=SelectFromModel(RandomForestClassifier(n_estimators=100,random_state=SEED,n_jobs=1),threshold=\'median\')\n    else: selector=\'passthrough\'\n    if balance==\'SMOTE\': sampler=SMOTE(random_state=SEED,k_neighbors=3)\n    elif balance==\'ADASYN\':sampler=ADASYN(random_state=SEED,n_neighbors=3)\n    else:sampler=\'passthrough\'\n    if algorithm==\'SVM\':\n        model=SVC(C=1.0,kernel=\'rbf\',gamma=\'scale\',probability=False,random_state=SEED,cache_size=512)\n    elif algorithm==\'Random Forest\':\n        model=RandomForestClassifier(n_estimators=200,max_depth=12,random_state=SEED,n_jobs=1)\n    elif algorithm==\'XGBoost\':\n        model=XGBClassifier(n_estimators=160,max_depth=3,learning_rate=0.1,\n          objective=\'binary:logistic\',eval_metric=\'logloss\',tree_method=\'hist\',\n          random_state=SEED,n_jobs=1)\n    else: raise ValueError(algorithm)\n    return Pipeline([(\'imputer\',SimpleImputer(strategy=\'median\',keep_empty_features=True)),\n        (\'scaler\',StandardScaler()),(\'selection\',selector),(\'sampler\',sampler),(\'model\',model)])\n\n\ndef raw_score(pipe,X):\n    if hasattr(pipe,\'predict_proba\'):\n        classes=list(pipe.classes_)\n        return np.asarray(pipe.predict_proba(X)[:,classes.index(1)],dtype=float),\'probabilidad_no_calibrada\'\n    if hasattr(pipe,\'decision_function\'):\n        return np.asarray(pipe.decision_function(X),dtype=float),\'margen_no_calibrado\'\n    raise ValueError(\'No hay score utilizable\')\n\n\ndef metrics(pipe,X,y):\n    y=np.asarray(y,dtype=int)\n    yp=np.asarray(pipe.predict(X),dtype=int)\n    s,kind=raw_score(pipe,X)\n    if not np.isfinite(s).all(): raise ValueError(\'Score no finito\')\n    cm=confusion_matrix(y,yp,labels=[0,1])\n    result={\'n\':len(y),\'positivos\':int(y.sum()),\'prevalencia\':float(y.mean()),\n       \'f1\':float(f1_score(y,yp,zero_division=0)),\n       \'average_precision\':float(average_precision_score(y,s)),\n       \'auc_roc\':float(roc_auc_score(y,s)) if len(np.unique(y))==2 else None,\n       \'precision\':float(precision_score(y,yp,zero_division=0)),\n       \'recall\':float(recall_score(y,yp,zero_division=0)),\n       \'accuracy\':float(accuracy_score(y,yp)),\n       \'tn\':int(cm[0,0]),\'fp\':int(cm[0,1]),\'fn\':int(cm[1,0]),\'tp\':int(cm[1,1]),\n       \'score_tipo\':kind}\n    return result,s,yp\n\n\ndef ranking_metrics(y,score,loc,fracs=(.1,.2,.3)):\n    y=np.asarray(y,dtype=int);n=len(y);pos=int(y.sum())\n    if pos==0: return []\n    idx=np.arange(n)\n    out=[]\n    ranks={\'ML\':np.lexsort((idx,-score)),\'LOC\':np.lexsort((idx,-np.asarray(loc,dtype=float)))}\n    for f in fracs:\n        k=max(1,int(np.ceil(n*f)))\n        for name,order in ranks.items():\n            true=int(y[order[:k]].sum())\n            out.append({\'metodo\':name,\'fraccion\':f,\'k\':k,\'detectados\':true,\n               \'precision_at_k\':true/k,\'recall_at_k\':true/pos,\n               \'lift_at_k\':(true/k)/float(y.mean())})\n        out.append({\'metodo\':\'Aleatorio_esperado\',\'fraccion\':f,\'k\':k,\'detectados\':None,\n            \'precision_at_k\':float(y.mean()),\'recall_at_k\':k/n,\'lift_at_k\':1.0})\n    return out\n\n\ndef config_hash(audits,projects):\n    info={\'version\':EXPERIMENT_VERSION,\'datasets\':{a[\'proyecto\']:a[\'sha256\'] for a in audits},\n       \'proyectos\':list(projects),\'candidatos\':list(CANDIDATES),\'seed\':SEED,\n       \'librerias\':{lib:version(lib) for lib in [\'scikit-learn\',\'imbalanced-learn\',\'xgboost\']}}\n    return hashlib.sha256(json.dumps(info,sort_keys=True).encode()).hexdigest(),info\n\n\ndef eval_one_candidate(data,train_projects,scenario,algorithm):\n    """Validación interna LOGO en PROYECTOS fuente; sin observar proyecto objetivo."""\n    folds=[];tic=time.time()\n    for validation_project in train_projects:\n        fit_projects=[p for p in train_projects if p!=validation_project]\n        X_tr,y_tr=pack(data,fit_projects)\n        X_val,y_val=data[validation_project]\n        pipe=make_pipeline(scenario,algorithm)\n        pipe.fit(X_tr,y_tr)\n        mm,_,_=metrics(pipe,X_val,y_val)\n        folds.append({\'proyecto_validacion\':validation_project,\n            **{k:mm[k] for k in [\'n\',\'positivos\',\'f1\',\'average_precision\',\'auc_roc\',\'precision\',\'recall\']}})\n    return {\'escenario\':scenario,\'algoritmo\':algorithm,\'n_folds\':len(folds),\n      \'f1_macro\':float(np.mean([d[\'f1\'] for d in folds])),\n      \'ap_macro\':float(np.mean([d[\'average_precision\'] for d in folds])),\n      \'auc_macro\':float(np.mean([d[\'auc_roc\'] for d in folds])),\n      \'segundos\':round(time.time()-tic,2),\'error\':None,\'folds\':folds}\n\n\ndef select_candidate(rows,expected_folds):\n    good=[r for r in rows if r.get(\'error\') is None and r.get(\'n_folds\')==expected_folds]\n    if not good:raise RuntimeError(\'Ningún candidato completo y válido\')\n    # Criterios fijados antes de ver proyectos reservados: F1 -> AP -> AUC.\n    return sorted(good,key=lambda r:(-r[\'f1_macro\'],-r[\'ap_macro\'],-r[\'auc_macro\'],\n                                     r[\'escenario\'],r[\'algoritmo\']))[0]\n\n\ndef inner_search(data,projects,checkpoint,lock,*,candidate_limit=None):\n    """Checkpoints por candidato (4 o 5 ajustes). Los errores se registran."""\n    checkpoint=Path(checkpoint)\n    if checkpoint.exists():\n        saved=json.loads(checkpoint.read_text(encoding=\'utf-8\'))\n        if saved[\'config_hash\']!=lock or saved[\'proyectos\']!=list(projects):\n            raise RuntimeError(f\'Checkpoint incompatible: {checkpoint}; no se mezclan experimentos\')\n        rows=saved[\'rows\']\n    else: rows=[]\n    done={(r[\'escenario\'],r[\'algoritmo\']) for r in rows}\n    candidates=CANDIDATES[:candidate_limit] if candidate_limit else CANDIDATES\n    for count,(scenario,algorithm) in enumerate(candidates,1):\n        if (scenario,algorithm) in done:continue\n        print(f\'  candidato {count:02}/{len(candidates)} · {algorithm} / {scenario}\',flush=True)\n        try:\n            row=eval_one_candidate(data,projects,scenario,algorithm)\n            print(f"      F1(macro)={row[\'f1_macro\']:.3f} · AP={row[\'ap_macro\']:.3f}",flush=True)\n        except Exception as exc:\n            row={\'escenario\':scenario,\'algoritmo\':algorithm,\'n_folds\':0,\n                 \'f1_macro\':None,\'ap_macro\':None,\'auc_macro\':None,\n                 \'segundos\':None,\'error\':f\'{type(exc).__name__}: {str(exc)[:400]}\',\'folds\':[]}\n            print(\'      ERROR:\',row[\'error\'],flush=True)\n        rows.append(row)\n        jdump(checkpoint,{\'config_hash\':lock,\'proyectos\':list(projects),\'rows\':rows})\n    return rows\n\n\ndef outer_test(data,target,selection):\n    sources=[p for p in data if p!=target]\n    Xt,yt=pack(data,sources)\n    Xo,yo=data[target]\n    pipe=make_pipeline(selection[\'escenario\'],selection[\'algoritmo\'])\n    pipe.fit(Xt,yt)\n    m,score,pred=metrics(pipe,Xo,yo)\n    # Sensibilidad a vectores X exactamente repetidos entre proyectos.\n    src_hash=set(pd.util.hash_pandas_object(Xt,index=False).to_numpy(dtype=np.uint64).tolist())\n    tgt_hash=pd.util.hash_pandas_object(Xo,index=False).to_numpy(dtype=np.uint64)\n    unseen=np.asarray([int(h) not in src_hash for h in tgt_hash],dtype=bool)\n    sensitivity=None\n    if unseen.sum()>1 and yo.iloc[unseen].nunique()==2:\n        yc=np.asarray(yo)[unseen];pc=pred[unseen];sc=score[unseen]\n        sensitivity={\'n\':int(unseen.sum()),\'positivos\':int(yc.sum()),\n           \'f1\':float(f1_score(yc,pc,zero_division=0)),\n           \'ap\':float(average_precision_score(yc,sc)),\n           \'auc\':float(roc_auc_score(yc,sc))}\n    m.update({\'proyecto_test\':target,\'proyectos_entrenamiento\':sources,\n       \'escenario\':selection[\'escenario\'],\'algoritmo\':selection[\'algoritmo\'],\n       \'cv_f1_macro_fuente\':selection[\'f1_macro\'],\n       \'vectores_test_presentes_en_train\':int((~unseen).sum()),\n       \'sensibilidad_sin_vectores_coincidentes\':sensitivity,\n       \'validation_protocolo\':\'nested_leave_one_project_out\'})\n    # Para el prototipo, ranking en proyecto reservado (etiq. solo para evaluación).\n    ranking=ranking_metrics(yo,score,Xo[\'loc\'].fillna(Xo[\'loc\'].median()).to_numpy())\n    for r in ranking:r[\'proyecto_test\']=target\n    return m,ranking\n\n\ndef run_nested(data,audits,results,*,candidate_limit=None):\n    """5 outer LOGO, cada uno 4 LOGO internos, luego selección final con 5 LOGO."""\n    results=Path(results);results.mkdir(parents=True,exist_ok=True)\n    names=list(data)\n    lock,config=config_hash(audits,names)\n    jdump(results/\'configuracion.json\',{\'hash\':lock,\'especificacion\':config,\n        \'advertencias\':[\'No generalización universal\',\'Proveniencia/alias pendientes de comprobación humana\',\n          \'F1 test independiente solo de la SELECCIÓN, no del pipeline final ajustado con todos los datos\',\n          \'Se debe comprobar solapamiento de vectores entre proyectos\']})\n    outer=[];ranking=[];algoritmos_outer=[]\n    for i,target in enumerate(names,1):\n        print(f\'\\n=== TEST EXTERNO {i}/{len(names)}: {target} ===\',flush=True)\n        source=[p for p in names if p!=target]\n        ck=results/f\'inner_{target}.json\'\n        rows=inner_search(data,source,ck,lock,candidate_limit=candidate_limit)\n        chosen=select_candidate(rows,len(source))\n        print(\'  Seleccionado por validación interna:\',chosen[\'algoritmo\'],chosen[\'escenario\'],flush=True)\n        # Cada algoritmo usa su mejor escenario interno, SIN elegir por el test externo.\n        for algoritmo in ALGORITHMS:\n            validos=[r for r in rows if r[\'algoritmo\']==algoritmo]\n            mejor_algo=select_candidate(validos,len(source))\n            metrica,_=outer_test(data,target,mejor_algo)\n            metrica[\'seleccionado_por_comparacion_interna_global\']=(\n                mejor_algo[\'algoritmo\']==chosen[\'algoritmo\'] and\n                mejor_algo[\'escenario\']==chosen[\'escenario\'])\n            algoritmos_outer.append(metrica)\n            print(f"   {algoritmo}: F1 externo {metrica[\'f1\']:.3f}",flush=True)\n            if metrica[\'seleccionado_por_comparacion_interna_global\']:\n                outer.append(metrica)\n                score_rank=metrica\n                # Repetimos exclusivamente el cálculo del ranking (sin selección de modelo).\n                X_train,y_train=pack(data,source)\n                pi=make_pipeline(chosen[\'escenario\'],chosen[\'algoritmo\'])\n                pi.fit(X_train,y_train)\n                X_test,y_test=data[target]\n                score,_=raw_score(pi,X_test)\n                ranks=ranking_metrics(y_test,score,X_test[\'loc\'].fillna(X_test[\'loc\'].median()).to_numpy())\n                for r in ranks:r[\'proyecto_test\']=target\n                ranking.extend(ranks)\n                jdump(results/f\'test_externo_{target}.json\',metrica)\n                pd.DataFrame(ranks).to_csv(results/f\'priorizacion_{target}.csv\',index=False)\n        pd.DataFrame([{k:v for k,v in row.items() if k!=\'folds\'} for row in rows]).to_csv(\n             results/f\'candidatos_inner_{target}.csv\',index=False)\n        # Guardar comparaciones acumuladas por si Colab se interrumpe.\n        pd.DataFrame(algoritmos_outer).drop(columns=[\'sensibilidad_sin_vectores_coincidentes\']).to_csv(\n             results/\'comparacion_algoritmos_outer_parcial.csv\',index=False)\n    pd.DataFrame(algoritmos_outer).drop(columns=[\'sensibilidad_sin_vectores_coincidentes\']).to_csv(\n         results/\'comparacion_algoritmos_outer.csv\',index=False)\n    comp=pd.DataFrame(algoritmos_outer).groupby(\'algoritmo\',as_index=False).agg(\n        f1_macro_externo=(\'f1\',\'mean\'),ap_macro_externo=(\'average_precision\',\'mean\'),\n        auc_macro_externo=(\'auc_roc\',\'mean\'),recall_macro_externo=(\'recall\',\'mean\'),\n        proyectos_evaluados=(\'proyecto_test\',\'nunique\'))\n    comp.sort_values([\'f1_macro_externo\',\'ap_macro_externo\'],ascending=False).to_csv(\n        results/\'resumen_algoritmos_externos.csv\',index=False)\n    # Métricas OUTER son evaluación anidada del procedimiento. No elegir algoritmo por Outer y reportar Outer\n    # como prueba independiente de ese algoritmo; el algoritmo final se selecciona por LOGO global.\n    jdump(results/\'resultados_outer.json\',outer)\n    pd.DataFrame(outer).drop(columns=[\'sensibilidad_sin_vectores_coincidentes\']).to_csv(\n         results/\'resultados_por_proyecto.csv\',index=False)\n    pd.DataFrame(ranking).to_csv(results/\'ranking_qa_todos.csv\',index=False)\n    print(\'\\n=== Selección final: CV entre los 5 proyectos históricos ===\',flush=True)\n    fullrows=inner_search(data,names,results/\'seleccion_final_cv.json\',lock,candidate_limit=candidate_limit)\n    winning=select_candidate(fullrows,len(names))\n    pd.DataFrame([{k:v for k,v in row.items() if k!=\'folds\'} for row in fullrows]).to_csv(\n        results/\'comparacion_global_cpdp.csv\',index=False)\n    valid=[r for r in fullrows if not r.get(\'error\')]\n    by_algo=pd.DataFrame(valid).groupby(\'algoritmo\',as_index=False).agg(\n        mejor_f1_macro=(\'f1_macro\',\'max\'),f1_promedio_9_escenarios=(\'f1_macro\',\'mean\'))\n    by_algo.to_csv(results/\'comparacion_algoritmos.csv\',index=False)\n    # El pipeline final ve los 5 proyectos después de que terminó el protocolo de selección.\n    X_all,y_all=pack(data,names)\n    pipe=make_pipeline(winning[\'escenario\'],winning[\'algoritmo\'])\n    pipe.fit(X_all,y_all)\n    modelpath=results/\'modelo_general_cpdp.joblib\'\n    manifest={\'protocolo\':EXPERIMENT_VERSION,\'nombre\':\'Modelo_general_CPDP\',\n        \'algoritmo\':winning[\'algoritmo\'],\'escenario\':winning[\'escenario\'],\n        \'seleccion_criterios\':\'F1_macro, AP_macro, AUC_macro en LOGO\',\n        \'cv_global_f1\':winning[\'f1_macro\'],\'cv_global_ap\':winning[\'ap_macro\'],\n        \'datasets_entrenamiento\':names,\'columnas\':list(FEATURES),\n        \'etiqueta\':\'1=defecto_historico_registrado\',\n        \'score_aviso\':\'SVM margen sin calibrar / RF o XGB probabilidad sin calibrar\',\n        \'documentally_verified\':False,\n        \'fuentes\':{a[\'proyecto\']:a[\'sha256\'] for a in audits},\n        \'algoritmo_final_elegido_por_logo_completo\':True,\n        \'limite\':\'El modelo final se entrenó con los cinco proyectos; sus métricas son de validación LOGO de procedimientos reentrenados, no una prueba inédita del .joblib final.\',\n        \'creado_utc\':datetime.now(timezone.utc).isoformat(),\n        \'advertencia_generalizacion\':\'Evaluación entre estos 5 proyectos; no asegura rendimiento universal\'}\n    bundle={\'pipeline\':pipe,\'manifest\':manifest,\'columnas\':list(FEATURES)}\n    joblib.dump(bundle,modelpath,compress=3)\n    jdump(results/\'manifiesto_modelo_general.json\',manifest)\n    # Plantilla para Streamlit: solo métricas+id, sin etiqueta.\n    pd.DataFrame(columns=[\'id_modulo\',*FEATURES]).to_csv(results/\'plantilla_para_streamlit.csv\',index=False)\n    summary={\'candidatos_por_busqueda\':len(CANDIDATES) if candidate_limit is None else candidate_limit,\n       \'folds_externos\':len(names),\'macro_f1_outer\':float(np.mean([r[\'f1\'] for r in outer])),\n       \'macro_ap_outer\':float(np.mean([r[\'average_precision\'] for r in outer])),\n       \'macro_auc_outer\':float(np.mean([r[\'auc_roc\'] for r in outer])),\n       \'ganador_final\':{\'algoritmo\':winning[\'algoritmo\'],\'escenario\':winning[\'escenario\'],\n         \'f1_cv_proyectos\':winning[\'f1_macro\'],\'ap_cv_proyectos\':winning[\'ap_macro\']},\n       \'errores_en_seleccion_final\':sum(r[\'error\'] is not None for r in fullrows),\n       \'nota\':\'La evaluación outer estima el procedimiento de selección, no es Test inédito del pipeline final.\'}\n    jdump(results/\'resumen_final.json\',summary)\n    return summary\n\n\ndef inference(bundle,df):\n    """Solo para prueba de integración; interfaz productiva será Streamlit."""\n    if \'id_modulo\' in df.columns:\n        ids=df[\'id_modulo\'].astype(str).tolist()\n    else:ids=[f\'modulo_{i+1:04}\' for i in range(len(df))]\n    X,aliases=harmonize_features(df.drop(columns=[\'id_modulo\'],errors=\'ignore\'), \'CSV externo\',allow_extra=False)\n    pipe=bundle[\'pipeline\']\n    score,kind=raw_score(pipe,X)\n    yp=pipe.predict(X)\n    out=pd.DataFrame({\'id_modulo\':ids,\'clase_predicha\':yp.astype(int),\n        \'score_riesgo\':score,\'tipo_score\':kind})\n    return out.sort_values(\'score_riesgo\',ascending=False,kind=\'stable\').reset_index(drop=True).assign(\n        prioridad=lambda x:np.arange(1,len(x)+1))\n\n\ndef descargar_datos_nuevos(raw_dir, timeout=75):\n    """Siempre descarga fuentes públicas; no lee carpetas ni resultados de versiones anteriores."""\n    import urllib.request\n    raw_dir=Path(raw_dir);raw_dir.mkdir(parents=True,exist_ok=True)\n    registros=[]\n    for p in PROJECTS:\n        url=f\'{SOURCE_BASE}/{p.lower()}.csv\'\n        destino=raw_dir/f\'{p}.csv\'\n        if destino.exists():\n            raise FileExistsError(f\'Para asegurar el inicio desde cero, la ruta ya existe: {destino}\')\n        request=urllib.request.Request(url,headers={\'User-Agent\':\'tesis-cpdp-reproducible/1.0\'})\n        with urllib.request.urlopen(request,timeout=timeout) as response:\n            blob=response.read(25_000_001)\n        if not (1000<len(blob)<=25_000_000):\n            raise RuntimeError(f\'{p}: tamaño de descarga sospechoso ({len(blob)})\')\n        destino.write_bytes(blob)\n        try:\n            _,_,aud=load_project(destino,p)\n        except Exception:\n            destino.unlink(missing_ok=True)\n            raise\n        registros.append({\'proyecto\':p,\'url\':url,\'archivo\':destino.name,\'sha256\':aud[\'sha256\'],\n                          \'registros\':aud[\'filas\'],\'variable_objetivo\':aud[\'etiqueta_original\']})\n        print(f\'{p}: {aud["filas"]} registros; SHA-256={aud["sha256"][:12]}...\')\n    return pd.DataFrame(registros)\n', encoding='utf-8')
sys.path.insert(0,str(BASE))
import cpdp_core_desde_cero as cpdp
importlib.reload(cpdp)
print('Módulo independiente cargado:', CORE_PY)
print('Escenarios:',len(cpdp.SCENARIOS),'| algoritmos:',len(cpdp.ALGORITHMS),'| combinaciones:',len(cpdp.CANDIDATES))

In [ ]:
fuentes = cpdp.descargar_datos_nuevos(RAW)
fuentes.to_csv(AUDIT/'fuentes_y_hashes.csv',index=False)
display(fuentes)
print('Se han descargado cinco CSV desde el repositorio público, NO desde Drive.')

## M02 — PREPROCESAMIENTO Y AUDITORÍA
- Comprobar etiquetas y las **21 métricas** en cada proyecto.
- Normalizar **solo los encabezados** (por ejemplo, `lOCodeAndComment` → `locCodeAndComment`); la equivalencia semántica sigue siendo una comprobación documental pendiente.
- Examinar prevalencia, faltantes, duplicados internos y **vectores idénticos entre proyectos**.
- **No** imputar, escalar, seleccionar características ni aplicar SMOTE/ADASYN globalmente: esas transformaciones se ajustan solo con datos de entrenamiento mediante un `Pipeline`.

**Precaución:** vectores iguales en diferentes proyectos pueden hacer que la evaluación parezca más favorable. El experimento informa el solapamiento y presenta una sensibilidad calculada sobre casos no coincidentes, sin ocultar ese límite.

In [ ]:
DATOS, auditorias = cpdp.load_all(RAW)
resumen = pd.DataFrame([{k:v for k,v in r.items() if k not in ('columnas_originales','alias_pendientes','procedencia_declarada')} for r in auditorias])
solapamiento=cpdp.overlap_audit(DATOS)
display(resumen[['proyecto','filas','positivos','negativos','prevalencia','variables','faltantes','duplicados_fila_completa','duplicados_vector_caracteristicas']])
print('Vectores idénticos entre proyectos:')
display(solapamiento)
resumen.to_csv(AUDIT/'resumen_datasets.csv',index=False)
solapamiento.to_csv(AUDIT/'solapamiento_entre_proyectos.csv',index=False)
cpdp.jdump(AUDIT/'auditoria_completa.json',auditorias)
assert len(DATOS)==5 and all(len(x.columns)==21 for x,y in DATOS.values())
print('Esquemas estructuralmente compatibles. Verificación documental de métricas y etiquetas: PENDIENTE.')

## M03 — PREDICTOR (algoritmos y escenarios)
**Clasificadores:** SVM, Random Forest y XGBoost.

**Escenarios:** 9 tratamientos que combinan ausencia de balanceo / SMOTE / ADASYN con ausencia de selección / Mutual Information / Random Forest.

El orden de cada pipeline es: imputación → escalado → selección de características → balanceo de la parte de entrenamiento → clasificador. Las operaciones están **dentro** de cada ajuste de validación, nunca realizadas usando el proyecto de prueba.

In [ ]:
display(pd.DataFrame(cpdp.SCENARIOS,columns=['escenario','balanceo','seleccion']))
print('Modelos:',cpdp.ALGORITHMS)
print('Total de configuraciones por búsqueda:',len(cpdp.CANDIDATES))
print('Criterio de selección fijado:', 'F1 macro por proyecto; desempate AP macro; luego AUC macro')

## M04 — VALIDACIÓN ENTRE PROYECTOS
**Diseño anidado para evitar elegir modelos mirando el Test:**

1. Se deja **un proyecto completo fuera** como Test externo.
2. Con los otros cuatro proyectos, se comparan **27 candidatos** mediante validación dejando uno de esos cuatro proyectos fuera cada vez.
3. Para cada algoritmo se selecciona internamente su mejor escenario y se mide en el proyecto externo. También se registra la mejor combinación global elegida **sin mirar ese proyecto externo**.
4. Se repite hasta evaluar los cinco proyectos como Test.

**Total esperado:** ~691 ajustes de modelos, más la exportación final. Puede tardar horas. El notebook guarda **checkpoint después de cada candidato** en su carpeta local nueva; si Colab borra la sesión, también se borran los archivos locales (descarga el ZIP después).

**No se promete un ganador universal.** La comparación entre algoritmos sobre cinco proyectos es una estimación limitada; la comparación de muchos métodos requiere interpretación y, si es viable, conjuntos externos adicionales.

In [ ]:
EJECUTAR_EXPERIMENTO = True
if EJECUTAR_EXPERIMENTO:
    resultado = cpdp.run_nested(DATOS, auditorias, EXP)
    print('EVALUACIÓN CPDP COMPLETADA')
    print(json.dumps(resultado,ensure_ascii=False,indent=2))
else:
    raise RuntimeError('El experimento está desactivado; cambia EJECUTAR_EXPERIMENTO=True')

## M05 — EVALUADOR Y MODELO FINAL
Lee exclusivamente los resultados recién creados por **este** notebook.

**Dos resultados distintos:**
- `resumen_algoritmos_externos.csv`: compara cada algoritmo en los cinco proyectos reservados, con su mejor escenario elegido internamente en cada ronda. Es una **comparación descriptiva entre proyectos**.
- `comparacion_global_cpdp.csv`: usa validación cruzada por proyecto en los cinco datasets históricos para seleccionar la configuración final, que después se **ajusta sobre todos esos datasets**.

La evaluación externa estima el procedimiento de entrenamiento/selección. **El archivo final `.joblib`, reajustado con los cinco datasets, no tiene un sexto proyecto inédito de Test**.

In [ ]:
obligatorios=[EXP/'resultados_por_proyecto.csv', EXP/'comparacion_algoritmos_outer.csv',
 EXP/'resumen_algoritmos_externos.csv',EXP/'comparacion_global_cpdp.csv',
 EXP/'modelo_general_cpdp.joblib', EXP/'manifiesto_modelo_general.json']
faltan=[p.name for p in obligatorios if not p.is_file() or p.stat().st_size==0]
if faltan: raise FileNotFoundError('NO hay entrenamiento científico completo; faltan: '+str(faltan))
print('Comparación de algoritmos en proyectos externos (elegidos internamente en cada ronda):')
display(pd.read_csv(EXP/'resumen_algoritmos_externos.csv'))
print('Evaluación externa del procedimiento seleccionado:')
display(pd.read_csv(EXP/'resultados_por_proyecto.csv')[[
 'proyecto_test','algoritmo','escenario','f1','average_precision','auc_roc',
 'recall','precision','vectores_test_presentes_en_train']])
print('Configuración única elegida con LOGO global:')
print((EXP/'manifiesto_modelo_general.json').read_text(encoding='utf-8'))

In [ ]:
# Gráfico de la comparación entre algoritmos sobre proyectos externos.
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(8,4))
comparacion=pd.read_csv(EXP/'resumen_algoritmos_externos.csv').sort_values('f1_macro_externo')
ax.barh(comparacion['algoritmo'],comparacion['f1_macro_externo'])
ax.set(xlabel='F1 macro entre proyectos reservados',title='Comparación entre algoritmos — CPDP',xlim=(0,1))
for i,val in enumerate(comparacion['f1_macro_externo']):ax.text(val+0.015,i,f'{val:.3f}',va='center')
fig.tight_layout()
fig.savefig(EVID/'f1_algoritmos_cpdp.png',dpi=150)
plt.show()

In [ ]:
# Prueba de carga e inferencia: solo verifica que el artefacto se pueda usar.
# No demuestra exactitud en un proyecto nuevo (se usa una muestra histórica para comprobar la API).
import joblib
paquete=joblib.load(EXP/'modelo_general_cpdp.joblib')
ejemplo=DATOS['PC1'][0].head(5).copy()
ejemplo.insert(0,'id_modulo',[f'prueba_{i+1:03}' for i in range(len(ejemplo))])
pred=cpdp.inference(paquete,ejemplo)
display(pred)
print('Pipeline final:',paquete['manifest']['algoritmo'], '+',paquete['manifest']['escenario'])
print('Los scores son NO calibrados; el CSV externo debe contener las 21 métricas compatibles.')
# Separar el artefacto listo para Streamlit del directorio experimental.
import shutil
shutil.copy2(EXP/'modelo_general_cpdp.joblib',FINAL/'modelo_general_cpdp.joblib')
shutil.copy2(EXP/'manifiesto_modelo_general.json',FINAL/'manifiesto_modelo_general.json')
shutil.copy2(EXP/'plantilla_para_streamlit.csv',FINAL/'plantilla_para_streamlit.csv')

## M06 — EXPORTADOR
Crea un único **ZIP completo** con: datos descargados, auditoría, comparaciones, métricas externas, ranking QA, pipeline final y el código autónomo. **No genera un ZIP vacío:** verifica los archivos obligatorios antes de exportar.

**Comparte este ZIP** para que podamos revisar los resultados; no hace falta volver a entrenar por cada consulta. No se monta ni se consulta Google Drive.

In [ ]:
import zipfile
obligatorios=[EXP/'resumen_final.json',EXP/'comparacion_global_cpdp.csv',
 EXP/'comparacion_algoritmos_outer.csv',EXP/'resultados_por_proyecto.csv',
 FINAL/'modelo_general_cpdp.joblib',FINAL/'manifiesto_modelo_general.json',
 AUDIT/'fuentes_y_hashes.csv']+[EXP/f'test_externo_{p}.json' for p in cpdp.PROJECTS]
faltan=[str(x) for x in obligatorios if not x.is_file() or x.stat().st_size==0]
if faltan: raise RuntimeError('No se exportó ZIP incompleto. Faltan: '+str(faltan))
ZIP=BASE/'Prediccion_Resultados.zip'
with zipfile.ZipFile(ZIP,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=5) as z:
    for folder in (RAW,AUDIT,EXP,FINAL,EVID):
        for p in folder.rglob('*'):
            if p.is_file() and not p.name.endswith('.tmp'):
                z.write(p,arcname=str(p.relative_to(BASE)))
    z.write(CORE_PY,arcname=CORE_PY.name)
with zipfile.ZipFile(ZIP) as z:
    bad=z.testzip()
    if bad:raise RuntimeError('ZIP dañado: '+str(bad))
    print('ZIP verificado. Archivos:',len(z.namelist()),'| MB:',round(ZIP.stat().st_size/1e6,2))
print('Archivo:',ZIP)
if 'google.colab' in sys.modules:
    from google.colab import files
    files.download(str(ZIP))

---
## Interpretación científica y límites

- El conjunto externo **se reserva por proyecto**, no por filas al azar.
- La métrica principal de selección es **F1 macro por proyecto**, con **Average Precision** y **AUC-ROC** para desempatar.
- No confundir el ganador del experimento con un modelo garantizado para **cualquier** proyecto.
- La entrada web será un **CSV con métricas compatibles**, no código fuente ni repositorios Git.
- La auditoría de fuente, semántica de etiquetas y duplicados entre proyectos debe explicarse en la tesis.
- Los archivos `.joblib` solo deben cargarse desde fuentes confiables; aquí fueron generados por tu propio notebook.

**Siguiente paso:** compartir `T2_CPDP_Desde_Cero_Resultados.zip`; revisaremos qué algoritmo fue mejor, la estabilidad por proyecto, las diferencias entre CV/Test y la integración con Streamlit.